# ejagruti
# TF-IDF: Find the right help article

**One realistic example:** An online shop has three help articles. A customer asks how to reset a password. We want to find the article that best matches that question.

We use only **scikit-learn** imports: `TfidfVectorizer` for text weights and `cosine_similarity` for comparing text. No pandas or NumPy imports, no manual TF-IDF implementation, and no dataset downloads.

**Run:** Select a Python kernel and choose **Run All**, or run each code cell with Shift+Enter. If scikit-learn is missing, run `%pip install scikit-learn` in a new cell, restart the kernel, and run from the top.

## 1. Import the ready-made tools

TF-IDF gives words numerical weights. A word used in fewer articles gets a higher IDF weight. Cosine similarity compares the resulting vectors; higher scores mean more overlap in their weighted word features.

In [1]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

## 2. Create our three short help articles

Each article is a **document**. Together, they form our **corpus** (collection of documents). The titles are only for displaying the result.

In [2]:
titles = ['Reset password', 'Track delivery', 'Return product']
articles = [
    'help reset your account password',
    'help track your order delivery',
    'help return your damaged product'
]

for title, article in zip(titles, articles):
    print(title + ': ' + article)

Reset password: help reset your account password
Track delivery: help track your order delivery
Return product: help return your damaged product


## 3. Convert the articles into TF-IDF numbers

`fit_transform()` learns the vocabulary and IDF weights from our articles, then converts each article into a vector.

A **vector** is a row of numbers. Each column belongs to one vocabulary word.

In [3]:
vectorizer = TfidfVectorizer()
article_vectors = vectorizer.fit_transform(articles)

print('Vocabulary (column order):')
print(vectorizer.get_feature_names_out())
print('\nTF-IDF matrix (one row per article):')
print(article_vectors.toarray().round(3))

Vocabulary (column order):
['account' 'damaged' 'delivery' 'help' 'order' 'password' 'product'
 'reset' 'return' 'track' 'your']

TF-IDF matrix (one row per article):
[[0.52  0.    0.    0.307 0.    0.52  0.    0.52  0.    0.    0.307]
 [0.    0.    0.52  0.307 0.52  0.    0.    0.    0.    0.52  0.307]
 [0.    0.52  0.    0.307 0.    0.    0.52  0.    0.52  0.    0.307]]


**How to read the output:** Rows follow the article order. Columns follow the printed vocabulary. Zero means a word is absent; a positive number is its TF-IDF weight.

`toarray()` and `round(3)` only make this tiny example readable. The calculation is done by scikit-learn. Keep large collections in their sparse form to save memory.

## 4. See why distinctive words receive higher IDF weights

`help` and `your` occur in all three articles. `password` occurs in only one. Inspect the IDF weights learned by the library.

In [4]:
for word, weight in zip(vectorizer.get_feature_names_out(), vectorizer.idf_):
    print(f'{word:10} IDF = {weight:.3f}')

account    IDF = 1.693
damaged    IDF = 1.693
delivery   IDF = 1.693
help       IDF = 1.000
order      IDF = 1.693
password   IDF = 1.693
product    IDF = 1.693
reset      IDF = 1.693
return     IDF = 1.693
track      IDF = 1.693
your       IDF = 1.000


You should see **1.000** for `help` and `your`, and approximately **1.693** for `password` and the other words that occur in only one article.

**Why are common words not zero, unlike our earlier textbook example?** Scikit-learn's defaults use smoothed IDF: `ln((1 + N) / (1 + DF)) + 1`. Here, N is the number of articles and DF is the number containing a word. It then scales each document vector to length 1 (L2 normalization). These choices explain why the printed TF-IDF values differ from simple textbook calculations. You do not need to implement this formula.

## 5. Convert the customer's question

Use `transform()` so the question uses the **same vocabulary and IDF weights**. Do not fit again on the question.

Words outside the article vocabulary are ignored. Here, `how`, `can`, and `my` are unseen; the default tokenizer also skips the one-letter `I`.

In [5]:
question = 'How can I reset my password?'
question_vector = vectorizer.transform([question])

print('Customer question:', question)
print('Question vector:')
print(question_vector.toarray().round(3))

Customer question: How can I reset my password?
Question vector:
[[0.    0.    0.    0.    0.    0.707 0.    0.707 0.    0.    0.   ]]


## 6. Compare the question with every article

`cosine_similarity()` returns one score per article. For these nonnegative TF-IDF vectors, scores range from 0 to 1. A score is a similarity measure, **not a probability or accuracy score**.

In [6]:
scores = cosine_similarity(question_vector, article_vectors)[0]

for title, score in zip(titles, scores):
    print(f'{title:16} similarity = {score:.3f}')

Reset password   similarity = 0.735
Track delivery   similarity = 0.000
Return product   similarity = 0.000


## 7. Show the best match

Python's `max()` finds the position of the largest score. If all scores are zero, we report no matching words instead of selecting an unrelated article.

In [7]:
best_index = max(range(len(scores)), key=lambda index: scores[index])

if scores[best_index] == 0:
    print('No matching words. Try a more specific question.')
else:
    print('Best article:', titles[best_index])
    print('Article text:', articles[best_index])

Best article: Reset password
Article text: help reset your account password


**Expected result:** `Reset password` is selected because `reset` and `password` match the first article. The other articles share no vocabulary words with this question, so their scores are zero.

## Try it yourself

Change only `question` in Step 5, then rerun Steps 5–7:

- `Where is my order delivery?` → Track delivery.
- `How do I return a damaged product?` → Return product.
- `banana` → No matching words.

## What did we learn?

**Articles → TF-IDF vectors → question vector → similarity scores → best article.**

Bag of Words uses counts; TF-IDF adjusts those counts using how widely words occur across the collection. This example is document retrieval, not a trained classifier. It matches words, not their deeper meaning: synonyms such as `parcel` and `delivery` are separate features. A classifier such as logistic regression could use TF-IDF features too, but would also need labeled training examples.

[Scikit-learn TF-IDF documentation](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html).